- Dataset & Dataloader Class

- Problems
    1) Memory inefficient due to batch gradient decent.
    2) Not Better Convergence due to batch gradient decent.

    - Solution:
        - Using batches (mini batch gradient descent) of data to train the model.

- The Dataset and Dataloader Classes
    - Dataset and Dataloader are core abstraction in PyTorch that decouple how you define your data from how you efficiently iterate over it in training loop. This means we sperate data load and data training process.
    - Consider we have csv data file having 10 rows and batch size is 2 so toatal batches = 10/2 = 5. 
        - We have to create custom dataset class which inherit from `CustomDataset(Dataset)`
    - Dataset Class:
        - The dataset class is essentially a blueprint when you create custom dataset, you decide how data is loaded and returned.
        - It defines:
            - `__init__()` which tells how data should be loaded.
            - `__len__()` which returns the total number of samples.
            - `__getitem__(index)` which returns the data (and labels) at given index

    - DataLoader Class:
        - The DataLoader wraps a Dataset and handles batching, shuffling and parallel loading.
    - DataLoader Control Flow:
        - At the start of each epoch, the DataLoader (if `shuffle=True`) shuffle indices(using a sampler)
        - It divides the indices into chunck of batch size.
        - For each index in the chunk, data samples are fetched from the Dataset object
        - The samples are then collected and combined into a batch (using collate function)
        - The batch is returned to the main training loop


In [52]:
from sklearn.datasets import make_classification
import torch

In [53]:
# step1: Create a synthetic classification dataset using sklearn
X , y = make_classification(
    n_samples= 10,          # number of samples
    n_features= 2,          # number of features
    n_informative=2,        # number of informative features
    n_redundant= 0,         # number of redundant fetures
    n_classes=2,            # number of classes
    random_state=42         # for reproducibiliy

)

In [54]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [55]:
X.shape

(10, 2)

In [56]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [57]:
y.shape

(10,)

In [58]:
# covert to tensor
X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y,dtype=torch.float32)

In [59]:
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [60]:
X.shape

torch.Size([10, 2])

In [61]:
y

tensor([1., 0., 0., 0., 0., 1., 1., 1., 1., 0.])

In [62]:
y.shape

torch.Size([10])

In [63]:
from torch.utils.data import Dataset, DataLoader

In [64]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [65]:
dataset = CustomDataset(X,y)

In [66]:
len(dataset)

10

In [67]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1.))

In [ ]:
dataloder = DataLoader(dataset,batch_size=2,shuffle=True,num_workers=5)

In [71]:
for batch_features, batch_labels in dataloder:
    print(batch_features)
    print(batch_labels)
    print("-"*70)

tensor([[-0.7206, -0.9606],
        [ 1.7273, -1.1858]])
tensor([0., 1.])
----------------------------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.1402, -0.8388]])
tensor([0., 0.])
----------------------------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-1.9629, -0.9923]])
tensor([0., 0.])
----------------------------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [ 1.0683, -0.9701]])
tensor([1., 1.])
----------------------------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.7774,  1.5116]])
tensor([1., 1.])
----------------------------------------------------------------------


- In PyTorch, the sampler in the DataLoader determines the strategy for selecting the dataset during data loading. It controls how indices of the dataset are drawn from each batch.

- Types of Samplers
    1) SequentialSampler:
        - Samples element sequentially, in the order they appear in the dataset.
        - Default when `shuffle=false`

    2) RandomSampler:
        - Samples elements randomly without replacement.
        - Defaukt when `shuffle=True`

- Collate Function (`collate_fn`):
    - The `collare-fn` in PyTorch DataLoader is a function that specifies how to combine a list of samples from a dataset into single batch. By default, the DataLoader uses a simple batch collation mechanism, but `collate_fn` allows you to customize how the data should be processed and batched.

Optimizing code

In [72]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [73]:
df = pd.read_csv("breast_cancer_data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [74]:
df.shape

(569, 33)

In [75]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

train test split

In [76]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

scaling

In [77]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Label Encoding

In [78]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

Numpy arrays to PyTorch tensors

In [79]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [80]:
from torch.utils.data import Dataset,DataLoader

class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [81]:
train_dataset = CustomDataset(X_test_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [83]:
train_dataset[10]

(tensor([-0.1001, -0.8173, -0.0696, -0.2085,  0.3408,  0.4592, -0.1097,  0.0627,
         -0.5579,  0.4598, -0.4439, -1.0693, -0.4330, -0.3664, -0.7955, -0.2396,
         -0.5052, -0.3830, -1.0072, -0.2356,  0.0600, -0.7945,  0.0902, -0.1221,
          0.4099,  0.6594,  0.0670,  0.3888, -0.5289,  1.1043]),
 tensor(1.))

In [85]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

Defining Model

In [87]:
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out

important patrameters

In [88]:
learning_rate = 0.1
epochs = 25

In [89]:
# create model
model = SimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

Training Pipeline

In [90]:
# define pipeline
for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

        # forward pass
        y_pred = model(batch_features)

        # loss calculate
        loss = loss_function(y_pred,batch_labels.view(-1,1))

        #clear gradients
        optimizer.zero_grad()

        # backward pass
        loss.backward()

        # parameter update
        optimizer.step()

    # print loss un each epoch
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 0.6547061204910278
Epoch: 2, Loss: 0.6205753087997437
Epoch: 3, Loss: 0.6519608497619629
Epoch: 4, Loss: 0.6136811375617981
Epoch: 5, Loss: 0.6632089018821716
Epoch: 6, Loss: 0.6627877950668335
Epoch: 7, Loss: 0.6347532272338867
Epoch: 8, Loss: 0.680484414100647
Epoch: 9, Loss: 0.6356436610221863
Epoch: 10, Loss: 0.672856867313385
Epoch: 11, Loss: 0.6063916683197021
Epoch: 12, Loss: 0.6049286723136902
Epoch: 13, Loss: 0.6147760152816772
Epoch: 14, Loss: 0.6539421677589417
Epoch: 15, Loss: 0.5882006883621216
Epoch: 16, Loss: 0.5033683776855469
Epoch: 17, Loss: 0.5783170461654663
Epoch: 18, Loss: 0.6446239352226257
Epoch: 19, Loss: 0.5748513340950012
Epoch: 20, Loss: 0.7111997008323669
Epoch: 21, Loss: 0.6309619545936584
Epoch: 22, Loss: 0.54915851354599
Epoch: 23, Loss: 0.5580279231071472
Epoch: 24, Loss: 0.526166558265686
Epoch: 25, Loss: 0.5604451298713684


In [91]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Accuracy: 0.5920
